# Phase 1 — Download and explore the Ms. Pac-Man data

**Goal of this notebook:** before building any model, *look at the data*. By the end you should be able to answer:

1. What does one frame of gameplay look like?
2. How are the joystick actions stored, and what does each number mean?
3. How often does the player press each direction? (Is the data balanced?)
4. What does a 16-frame clip (the thing our model will see) look like?

**Why this matters:** the model can only learn what the labels say. If we misunderstand the labels, every later step is wrong and nothing will crash to warn us. Ten minutes here saves a week later.

Run the cells top to bottom. Write anything surprising in `notes.md`.

## 0. Check your machine

This tells us whether you have a GPU. You don't need one for this notebook. It only matters for training (Phase 4).

In [ ]:
import platform, sys
print("Python:", sys.version.split()[0])
print("OS:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__)
    print("NVIDIA GPU (cuda):", torch.cuda.is_available())
    print("Apple GPU (mps):  ", torch.backends.mps.is_available())
except ImportError:
    print("torch not installed yet. Run:  %pip install torch")

## 1. Install the libraries this notebook needs

In [ ]:
%pip install -q numpy pillow matplotlib requests tqdm

In [ ]:
import os, re, tarfile, zipfile
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import requests
from PIL import Image
from tqdm.auto import tqdm

DATA_DIR = "data/raw"          # everything big goes here (it's in .gitignore)
os.makedirs(DATA_DIR, exist_ok=True)

## 2. Download the dataset

We use **Atari-HEAD** (Atari Human Eye-Tracking And Demonstration), hosted on Zenodo. People played Atari games while their screen, joystick and eye movements were recorded. We only need the screen frames and the joystick actions.

Two files:
- `action_enums.txt` (tiny): the table that says what each action number means.
- `ms_pacman.zip` (**about 5 GB**): the gameplay. Use a good Wi-Fi connection. If the download stops partway, just re-run the cell and it skips files you already have.

In [ ]:
RECORD_ID = "3451402"
WANTED = ["action_enums.txt", "ms_pacman.zip"]

files = requests.get(f"https://zenodo.org/api/records/{RECORD_ID}", timeout=30).json()["files"]
by_name = {f["key"]: f for f in files}
print("Files in this record:", sorted(by_name))

def download(name):
    f = by_name[name]
    out = os.path.join(DATA_DIR, name)
    if os.path.exists(out) and os.path.getsize(out) == f["size"]:
        print(f"already have {name}")
        return out
    with requests.get(f["links"]["self"], stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(out, "wb") as fh, tqdm(total=f["size"], unit="B", unit_scale=True, desc=name) as bar:
            for chunk in r.iter_content(1 << 20):
                fh.write(chunk)
                bar.update(len(chunk))
    return out

for name in WANTED:
    download(name)

## 3. What do the action numbers mean?

**This is the most important cell in the notebook.** The repo we're learning from assumed `1 = UP, 2 = RIGHT, ...`. Let's see what the dataset itself says.

In [ ]:
print(open(os.path.join(DATA_DIR, "action_enums.txt")).read())

**What to check:** find which number is `UP`, `RIGHT`, `LEFT`, `DOWN`. If `1` is `FIRE` and `2` is `UP`, the original repo's labels were shifted by one. Write down what you see in `notes.md`.

We'll keep a Python list of the names so we never hard-code numbers again. **If the printout above disagrees with this list, fix the list to match the file.**

In [ ]:
ALE_ACTIONS = ['NOOP', 'FIRE', 'UP', 'RIGHT', 'LEFT', 'DOWN',
               'UPRIGHT', 'UPLEFT', 'DOWNRIGHT', 'DOWNLEFT',
               'UPFIRE', 'RIGHTFIRE', 'LEFTFIRE', 'DOWNFIRE',
               'UPRIGHTFIRE', 'UPLEFTFIRE', 'DOWNRIGHTFIRE', 'DOWNLEFTFIRE']
for i, a in enumerate(ALE_ACTIONS):
    print(i, a)

## 4. Unzip and find the game sessions ("trials")

Each **trial** is one ~15-minute play session. It comes as two files:
- `<trial>.tar.bz2`: a compressed folder of PNG images, one per frame
- `<trial>.txt`: a table with one row per frame (including which action was pressed)

In [ ]:
zip_path = os.path.join(DATA_DIR, "ms_pacman.zip")
extract_to = os.path.join(DATA_DIR, "ms_pacman")
if not os.path.exists(extract_to):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_to)

# find every trial (a .txt with a matching .tar.bz2 next to it), wherever the zip put them
trials = []
for root, _, fnames in os.walk(extract_to):
    for fn in fnames:
        if fn.endswith(".tar.bz2"):
            stem = fn[:-len(".tar.bz2")]
            if os.path.exists(os.path.join(root, stem + ".txt")):
                trials.append((root, stem))
trials.sort()
print(f"Found {len(trials)} trials")
for root, stem in trials[:5]:
    print("  ", os.path.relpath(root, extract_to), stem)

## 5. Read one trial's label file

Let's look at the raw text first. Never trust a format you haven't seen.

In [ ]:
root, stem = trials[0]
txt_path = os.path.join(root, stem + ".txt")
with open(txt_path) as f:
    for _ in range(4):
        print(f.readline().rstrip()[:200])

The first line is the header. The columns we care about:
- `frame_id`: which image this row belongs to
- `action`: the joystick action (column index 5)

The rest (score, reaction time, gaze positions) we ignore.

In [ ]:
def load_labels(txt_path):
    """Return a list of (frame_id, action_int). Rows with a missing action are skipped."""
    rows = []
    with open(txt_path) as f:
        f.readline()  # header
        for line in f:
            parts = line.strip().split(",")
            if len(parts) > 5:
                try:
                    rows.append((parts[0], int(parts[5])))
                except ValueError:
                    pass   # e.g. 'null' actions
    return rows

labels = load_labels(txt_path)
print(f"{len(labels):,} labeled frames in {stem}")
print(labels[:5])

## 6. How often is each action pressed?

Counting over **all** trials tells us whether the data is balanced. If one action makes up most of the data, a lazy model can score well by always guessing it.

In [ ]:
counts = Counter()
for root, stem in tqdm(trials, desc="counting"):
    counts.update(a for _, a in load_labels(os.path.join(root, stem + ".txt")))

total = sum(counts.values())
print(f"{total:,} frames total\n")
for a, n in counts.most_common():
    print(f"{a:2d}  {ALE_ACTIONS[a]:14s} {n:>9,}  {100*n/total:5.1f}%")

In [ ]:
ids = sorted(counts)
plt.figure(figsize=(10, 3.5))
plt.bar([ALE_ACTIONS[a] for a in ids], [counts[a] for a in ids], color="#4a7bd0")
plt.xticks(rotation=60, ha="right"); plt.ylabel("frames"); plt.title("Ms. Pac-Man: how often each action was pressed")
plt.tight_layout(); plt.show()

**Questions to answer in `notes.md`:**
- Which four actions dominate? Do they match `UP / RIGHT / LEFT / DOWN` using the table from step 3?
- How rare are the diagonals? (A 1-in-100 class is hard to learn. Remember this for Phase 5.)
- How much is `NOOP`?

## 7. Look at actual frames

Extract the first trial's images and match each image to its label **by name** (`frame_id`). Matching by name is safer than sorting files and hoping the order lines up.

In [ ]:
root, stem = trials[0]          # (the counting loop above reused these names)
frames_dir = os.path.join(root, stem)
if not os.path.exists(frames_dir):
    with tarfile.open(os.path.join(root, stem + ".tar.bz2"), "r:bz2") as tar:
        tar.extractall(root)

pngs = {}
for r, _, fnames in os.walk(frames_dir):
    for fn in fnames:
        if fn.endswith(".png"):
            pngs[fn[:-4]] = os.path.join(r, fn)

labels = load_labels(os.path.join(root, stem + ".txt"))
matched = [(pngs[fid], a) for fid, a in labels if fid in pngs]
print(f"{len(pngs):,} images, {len(labels):,} labels, {len(matched):,} matched")

If `matched` is much smaller than the other two, the `frame_id` doesn't equal the image file name. Print a few of each and compare: `list(pngs)[:3]` and `labels[:3]`.

In [ ]:
img = Image.open(matched[1000][0])
print("Original size (width x height):", img.size, "mode:", img.mode)
fig, ax = plt.subplots(1, 2, figsize=(8, 5))
ax[0].imshow(img); ax[0].set_title("original")
small = img.convert("L").resize((84, 84), Image.BILINEAR)
ax[1].imshow(small, cmap="gray"); ax[1].set_title("what the model sees: 84x84 gray")
for a in ax: a.axis("off")
plt.show()

**Why shrink to 84×84 grayscale?** It's ~20× less data per frame, and color barely matters for "which way did Pac-Man move." 84×84 is the standard size from DeepMind's Atari work.

## 8. See a training example: a 16-frame clip

This is exactly what the model will get as input: **16 frames, every 4th frame**, labeled with the action at **frame #9 (index 8)**, the middle one. The model sees 8 frames before that moment and 7 after. Seeing the "after" is what makes guessing the button possible. That's the core idea of an inverse dynamics model.

In [ ]:
SEQ, SKIP = 16, 4

def show_clip(start):
    idxs = [start + j * SKIP for j in range(SEQ)]
    mid = idxs[SEQ // 2]
    fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
    for k, (ax, i) in enumerate(zip(axes.flat, idxs)):
        ax.imshow(Image.open(matched[i][0]).convert("L").resize((84, 84)), cmap="gray")
        ax.set_title(("LABEL " if i == mid else "") + f"#{k}", color="red" if i == mid else "black", fontsize=9)
        ax.axis("off")
    fig.suptitle(f"label = {ALE_ACTIONS[matched[mid][1]]}  (raw id {matched[mid][1]})", fontsize=13)
    plt.tight_layout(); plt.show()

show_clip(2000)

## 9. Sanity check: does the label match the motion?

Pick clips whose label is a single direction and watch Pac-Man around the red frame. If the label says `UP`, Pac-Man should be heading up. **This is how you confirm (or rule out) the off-by-one bug with your own eyes.**

In [ ]:
import random
target = ALE_ACTIONS.index("UP")   # try "LEFT", "RIGHT", "DOWN" too
candidates = [s for s in range(0, len(matched) - SEQ * SKIP)
              if matched[s + (SEQ // 2) * SKIP][1] == target]
print(f"{len(candidates):,} clips labeled {ALE_ACTIONS[target]}")
for s in random.sample(candidates, 2):
    show_clip(s)

## Done with Phase 1

Write in `notes.md`:
1. What the action numbers mean (from step 3), and whether the motion in step 9 agreed.
2. The action distribution (step 6): what dominates, what's rare.
3. Anything weird (unmatched frames, null actions, odd image sizes).

Next is **Phase 2**: turning every trial into thousands of these 16-frame clips and saving them so the model can train on them.